BGE-m3-multivector-initialization

In [ ]:
from FlagEmbedding import BGEM3FlagModel
from pymilvus import (
    MilvusClient,connections,FieldSchema,CollectionSchema,DataType,Collection,RRFRanker,utility,AnnSearchRequest
)
from datasets import Dataset
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List,Dict,Tuple
from datasets import load_dataset

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"]=(12,6)

/Users/nilasark/advanced/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:


documents = [
    "Global warming is primarily caused by increased greenhouse gas emissions from human activities, particularly the burning of fossil fuels like coal, oil, and natural gas.",
    "Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.",
    "The jet stream forms a boundary between the cold north and the warmer south, but the lower temperature difference means the winds are now weaker, leading to more extreme weather patterns.",
    "Coral reefs become stressed due to ocean acidification and warming, expelling their symbiotic algae which leaves the coral a bleached white color. This process threatens entire marine ecosystems.",
    "The rapid changes in the climate may have profound consequences for humans and other species. Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting in intense and widespread forest fires.",
    "Rising sea levels threaten coastal cities worldwide, with predictions suggesting that many major urban centers could face significant flooding by 2100 if current trends continue.",
    "Melting Arctic ice reduces the Earth's albedo effect, causing the planet to absorb more solar radiation and accelerating the warming process in a dangerous feedback loop.",
    "Climate change is disrupting agricultural patterns, forcing farmers to adapt their crops and techniques to new temperature and precipitation regimes that differ from historical norms."
]

print(f"Loaded {len(documents)} documents")
print(f"Sample Documents")
for idx,doc in enumerate(documents,1):
    print(f"{idx}: {doc[:200]}")



Loaded 8 documents
Sample Documents
1: Global warming is primarily caused by increased greenhouse gas emissions from human activities, particularly the burning of fossil fuels like coal, oil, and natural gas.
2: Over the coming 25 or 30 years, scientists say, the climate is likely to gradually warm. However, researchers also say that this phenomenon can be stopped if human emissions are reduced to zero.
3: The jet stream forms a boundary between the cold north and the warmer south, but the lower temperature difference means the winds are now weaker, leading to more extreme weather patterns.
4: Coral reefs become stressed due to ocean acidification and warming, expelling their symbiotic algae which leaves the coral a bleached white color. This process threatens entire marine ecosystems.
5: The rapid changes in the climate may have profound consequences for humans and other species. Severe drought caused food shortages for millions of people in Ethiopia, with a lack of rainfall resulting

In [4]:
print("Loading BGE-M3 model ...")

model=BGEM3FlagModel(
    model_name_or_path="BAAI/bge-m3",
    devices="cpu"
)
print("BGE-M3 model successfully loaded")

Loading BGE-M3 model ...


Fetching 30 files: 100%|██████████| 30/30 [00:00<00:00, 371177.35it/s]


BGE-M3 model successfully loaded


BGE-m3 embeddings generated

In [7]:
print(f"Sparse,Dense and ColBERT embeddings will be genrated")

doc_embeddings=model.encode(
    documents,
    batch_size=1000,
    max_length=512,
    return_dense=True,
    return_sparse=True,
    return_colbert_vecs=False
)
dense_embeddings=doc_embeddings['dense_vecs']
sparse_embeddings=doc_embeddings['lexical_weights']

print(f"Dense Vector Shape : {dense_embeddings.shape}")
print(f"Sparse Embedding Size: {len(sparse_embeddings)} length")



Sparse,Dense and ColBERT embeddings will be genrated
Dense Vector Shape : (8, 1024)
Sparse Embedding Size: 8 length


Milvus BGE-m3 Collection created

In [16]:
connections.connect(
    alias="default",
    uri="http://localhost:19530"
)

print(f"Connected To Milvus")
collection_name="bge_m3_hybrid"

if utility.has_collection(collection_name):
    utility.drop_collection(collection_name)
    print(f"Dropping Collection")

fields=[
    FieldSchema(name='pk',dtype=DataType.VARCHAR,is_primary=True,auto_id=True,max_length=100),
    FieldSchema(name="text",dtype=DataType.VARCHAR,max_length=2000),
    FieldSchema(name="dense_vector",dtype=DataType.FLOAT_VECTOR,dim=1024),
    FieldSchema(name='sparse_vector',dtype=DataType.SPARSE_FLOAT_VECTOR)
]

schema=CollectionSchema(
    fields=fields,
    description="BGE-M3 multi-vector hybrid retrieval demo"
)

collection=Collection(
    name=collection_name,
    schema=schema
)

print(f"Collection {collection_name} created")

Connected To Milvus
Collection bge_m3_hybrid created


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1890051003.py:1: PyMilvusDeprecationWarning: `connections.connect` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  connections.connect(
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1890051003.py:9: PyMilvusDeprecationWarning: `utility.has_collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  if utility.has_collection(collection_name):
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1890051003.py:25: PyMilvusDeprecationWarning: `Collection` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection=Collection(


Index Creation

In [18]:
print("Creating Indexes ...")

dense_index_params={
    "index_type":"IVF_FLAT",
    "metric_type": "IP",
    "params":{"nlist": 64}
}
collection.create_index(
    field_name="dense_vector",
    index_params=dense_index_params
)
print(f"Dense Index Created")

sparse_index_params={
    "index_type": "SPARSE_INVERTED_INDEX",
    "metric_type": "IP"
}
collection.create_index(
    field_name="sparse_vector",
    index_params=sparse_index_params
)
print(f"Sparse Index Created ")

collection.load()
print(f"Collection Loaded Into Memory")

Creating Indexes ...


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/2937074485.py:8: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(


Dense Index Created


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/2937074485.py:18: PyMilvusDeprecationWarning: `Collection.create_index` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.create_index(


Sparse Index Created 


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/2937074485.py:24: PyMilvusDeprecationWarning: `Collection.load` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.load()


Collection Loaded Into Memory


Data Insertion

In [19]:
print(f"Inseting Documents in the MILVUS")

entities=[
    documents,
    dense_embeddings,
    sparse_embeddings
]

collection.insert(entities)
print(f"Inserted {len(documents)} in the Disk")
collection.flush()
print(f"Data Flushed to the Disk")

Inseting Documents in the MILVUS
Inserted 8 in the Disk


/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/976005418.py:9: PyMilvusDeprecationWarning: `Collection.insert` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.insert(entities)
/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/976005418.py:11: PyMilvusDeprecationWarning: `Collection.flush` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  collection.flush()


Data Flushed to the Disk


Dense Search 

In [ ]:
def dense_search(query:str,limit:int=5)->List[Dict]:
    query_embedding=model.encode(
        query,
        return_dense=True,
        return_colbert_vecs=False,
        return_sparse=True
    )

    dense_vector=query_embedding['dense_vecs']
    results=collection.search(
        data=[dense_vector.tolist()],
        anns_field="dense_vector",
        limit=limit,
        param={"metric_type":"IP","params":{"nprobe": 10}},
        output_fields=["text"]
    )

    formatted_result=[]
    rank=1
    for hit in results:
        for hits in hit:
            formatted_result.append(
                {
                    "text":hits['entity']['text'],
                    "score":hits.score,
                    "rank":rank
                }
            )
            rank=rank+1

    return formatted_result

/var/folders/j_/jb1h_wtd21zfymdrgw0tg8500000gn/T/ipykernel_19987/1064341028.py:10: PyMilvusDeprecationWarning: `Collection.search` is an ORM-style PyMilvus API and will be removed in PyMilvus 3.1. Use `MilvusClient` instead.
  results=collection.search(


Sparse Search

In [ ]:
def sparse_search(query:str,limit:int=5)->List[Dict]:
    query_embeddings=model.encode(
        query,
        return_dense=False,
        return_sparse=True,
        return_colbert_vecs=False
    )
    sparse_embeddings=query_embeddings['lexical_weights']
    result=collection.search(
        [sparse_embeddings],
        anns_field="sparse_vector",
        limit=limit,
        param={"metric_type":"IP"},
        output_fields=["text"]
    )


    formatted_result={}
    rank=1
    for hits in result:
        for hit in hits:
            formatted_result.append({
                "Document":hit['entity']['text'],
                "score":hit.score,
                "rank":rank
            })
            rank=rank+1
    return formatted_result

Hybrid Search

In [ ]:
def hybrid_search(query:str,limit:int=5)->List[Dict]:
    query_embedding=model.encode(
        query,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False
    )

    dense_vector=query_embedding['dense_vecs']
    sparse_vector=query_embedding['lexical_weights']

    dense_search_params={"metric_type":"IP","params":{"nprobe":10}}
    sparse_search_params={"metric_type":"IP"}

    dense_result=AnnSearchRequest(
        [dense_vector.tolist()],
        anns_field="dense_vector",
        param=dense_search_params,
        limit=limit
    )

    sparse_result=AnnSearchRequest(
         [sparse_vector],
         anns_field="sparse_vector",
         param=dense_search_params,
         limit=limit
    )

    results=collection.hybrid_search(
        reqs=[dense_result,sparse_result],
        rerank=RRFRanker,
        limit=limit,
        output_fields=["text"]
    )

    formatted_result={}
    rank=1
    for hits in results:
        for hit in hits:
            formatted_result.append({
                "Document":hit['entity']['text'],
                "score":hit.score,
                "rank":rank
            })
            rank=rank+1
    return formatted_result

In [ ]:
def rerank_with_colbert(query:str,candidate_texts:List[str],top_k:int=3)->List[dict]:
    query_embeddings=model.encode(
        query,
        return_dense=False,
        return_sparse=False,
        return_colbert_vecs=True
    )
    query_vector=query_embeddings['colbert_vecs']

    candidate_embeddings=model.encode(
        candidate_texts,
         return_dense=False,
         return_sparse=False,
         return_colbert_vecs=True
    )

    candidate_vector=candidate_embeddings['colbert_vecs']
    scores=[]
    for vectors in candidate_vector:
        similarity=np.dot(query_vector,vectors.T)
        max_sims=np.max(similarity,axis=1)
        scores.append(np.mean(max_sims))
    rank_indices=np.argsort(scores)[::-1][:top_k]
    ranked_result=[]
    for rank,idx in enumerate(rank_indices,1):
        ranked_result.append({
            "text":candidate_texts[idx],
            "score": float(scores[idx]),
            "rank": rank
        })

    return ranked_result